In [1]:
import os
import pickle
import json
import time

from pyboolnet.external.bnet2primes import bnet_text2primes
from pystablemotifs.format import primes2bnet

from boolmore.core.conversions import merge_rules, prime2rr
from boolmore.eval.constraint import check_node

In [2]:
MODELS_DIR = "./target_models"

json_file = "./Tcell_config.json"

CACHE_FILE = "Tcell_primes.pkl"

OUTPUT_FILE = "Tcell_merged.bnet"

In [3]:
# read every file that ends with .bnet in the current directory
files = [f for f in os.listdir(MODELS_DIR) if f.endswith('.bnet')]
print(files)

['P18_BBM66.bnet', 'MX_BBM40.bnet', 'MS15_BBM62.bnet', 'M06_BBM154.bnet', 'MSM_BBM17.bnet', 'AJ_BBM157.bnet', 'N10_BBM155.bnet']


In [4]:
primes_list = []
for f in files:
    with open(MODELS_DIR + '/' + f, 'r') as file:
        
        # if f not in ['MX_BBM40.bnet', 'MS15_BBM62.bnet', 'M06_BBM154.bnet']:
        #     continue

        print("importing", f)
        bnet_text = file.read()
        primes = bnet_text2primes(bnet_text)

        n = len(primes)
        sources = []
        for node in primes:
            if primes[node] == [[{node:0}],[{node:1}]]:
                sources.append(node)

        print(f"\tnumber of nodes: {n}")
        print(f"\tnumber of source nodes: {len(sources)}")

        primes_list.append(primes)


importing P18_BBM66.bnet
	number of nodes: 39
	number of source nodes: 10
importing MX_BBM40.bnet
	number of nodes: 23
	number of source nodes: 4
importing MS15_BBM62.bnet
	number of nodes: 18
	number of source nodes: 6
importing M06_BBM154.bnet
	number of nodes: 21
	number of source nodes: 3
importing MSM_BBM17.bnet
	number of nodes: 50
	number of source nodes: 9
importing AJ_BBM157.bnet
	number of nodes: 82
	number of source nodes: 20
importing N10_BBM155.bnet
	number of nodes: 58
	number of source nodes: 13


In [5]:
merged_primes, merged_regulators_dict, merged_rr_dict, merged_signs_dict = merge_rules(primes_list)

print(merged_primes)

{'APC': [[{'APC': 0}], [{'APC': 1}]], 'BCL6': [[{'IL2': 1, 'STAT1': 0, 'STAT3': 0, 'STAT4': 0}, {'IL2': 1, 'STAT3': 0, 'STAT5': 1}, {'IL2': 1, 'STAT4': 0, 'STAT5': 1}, {'IL21': 0, 'STAT1': 0, 'STAT3': 0, 'STAT4': 0}, {'IL21': 0, 'STAT3': 0, 'STAT5': 1}, {'IL21': 0, 'STAT4': 0, 'STAT5': 1}, {'STAT1': 0, 'STAT3': 0, 'STAT4': 0, 'TGFB': 1}, {'STAT3': 0, 'STAT5': 1, 'TGFB': 1}, {'STAT4': 0, 'STAT5': 1, 'TGFB': 1}, {'TBET': 1}], [{'IL2': 0, 'IL21': 1, 'TBET': 0, 'TGFB': 0}, {'STAT1': 1, 'STAT5': 0, 'TBET': 0}, {'STAT3': 1, 'STAT4': 1, 'TBET': 0}, {'STAT3': 1, 'STAT5': 0, 'TBET': 0}, {'STAT4': 1, 'STAT5': 0, 'TBET': 0}]], 'CD28': [[{'APC': 0}], [{'APC': 1}]], 'CD4': [[{'CD4': 0, 'NOTCH1': 0, 'THPOK': 0}, {'NOTCH1': 0, 'RUNX3': 1, 'THPOK': 0}], [{'CD4': 1, 'RUNX3': 0}, {'NOTCH1': 1}, {'THPOK': 1}]], 'CD8': [[{'CD8': 0, 'NOTCH1': 0, 'RUNX3': 0}, {'TCR': 1}, {'THPOK': 1}], [{'CD8': 1, 'TCR': 0, 'THPOK': 0}, {'NOTCH1': 1, 'TCR': 0, 'THPOK': 0}, {'RUNX3': 1, 'TCR': 0, 'THPOK': 0}]], 'CMAF': [[{'S

In [6]:
# def my_main_function():
#     merged_primes, regulators, _, _ = merge_rules(primes_list)

# # Just type this at the bottom of the cell (no imports needed)
# %prun my_main_function()

In [7]:
# override = """
# IL23R,          (IL23 | IL23_e | RORGT | STAT3) & !TBET
# IL4R,           (IL4 | IL4R&IL4R_2 | IL4_e) & !SOCS1
# RORGT,          (!BCL6&!FOXP3&!GATA3&IL21&!TBET&TGFB | !FOXP3&!GATA3&RORGT&!TBET | RORGT&STAT3 | RORGT&TGFBR | SMAD2&STAT3 | STAT3&TGFBR) & IL17
# """

# override_primes = bnet_text2primes(override)



# for node in override_primes:
#     if override_primes[node] == [[{node:0}],[{node:1}]]:
#         continue
#     regulators, rr, signs = prime2rr(override_primes[node], regulators=None, signs=None)
        
#     print("overriding", node)
#     merged_primes[node] = override_primes[node]
#     merged_regulators_dict[node] = regulators
#     merged_rr_dict[node] = rr
#     merged_signs_dict[node] = signs

In [8]:
f = open(json_file)
json_dict = json.load(f)

constraints = json_dict["constraints"]

check = True
for node in merged_primes:
    start = time.perf_counter()

    check = check_node(merged_regulators_dict[node],
                       merged_rr_dict[node],
                       merged_rr_dict[node],
                       constraints,
                       node) and check

    end = time.perf_counter()
    print(f"Checking {node} took {end - start:.6f} seconds.")

Checking APC took 0.000012 seconds.
Checking BCL6 took 0.001088 seconds.
Checking CD28 took 0.000016 seconds.
Checking CD4 took 0.000070 seconds.
Checking CD8 took 0.000079 seconds.
Checking CMAF took 0.000013 seconds.
Checking DLL1 took 0.000005 seconds.
Checking EOMES took 0.000019 seconds.
STAT6 should regulate FOXP3
Checking FOXP3 took 2.449289 seconds.
NFAT should regulate GATA3
Checking GATA3 took 0.766970 seconds.
Checking GZMB took 0.000029 seconds.
Checking IFNAR took 0.000019 seconds.
Checking IFNA_e took 0.000005 seconds.
Checking IFNBR took 0.000014 seconds.
Checking IFNB_e took 0.000002 seconds.
RUNX3 should regulate IFNG
Checking IFNG took 34.492087 seconds.
NFAT should regulate IFNGR
Checking IFNGR took 0.000095 seconds.
Checking IFNGR_2 took 0.000005 seconds.
Checking IFNG_2 took 0.000008 seconds.
Checking IFNG_e took 0.000001 seconds.
Checking IKB took 0.000006 seconds.
Checking IL10 took 0.264003 seconds.
Checking IL10R took 0.000029 seconds.
Checking IL10_e took 0.00

In [9]:
with open(CACHE_FILE, "wb") as f:
    pickle.dump(merged_primes, f)
print("Computed and cached primes.")

Computed and cached primes.


In [10]:
merged_bnet = primes2bnet(merged_primes)
print(merged_bnet)

with open(OUTPUT_FILE, "w") as f:
    f.write(merged_bnet)
print(f"Saved merged model to {OUTPUT_FILE}")

APC,            APC
BCL6,           !IL2&IL21&!TBET&!TGFB | STAT1&!STAT5&!TBET | STAT3&STAT4&!TBET | STAT3&!STAT5&!TBET | STAT4&!STAT5&!TBET
CD28,           APC
CD4,            CD4&!RUNX3 | NOTCH1 | THPOK
CD8,            CD8&!TCR&!THPOK | NOTCH1&!TCR&!THPOK | RUNX3&!TCR&!THPOK
CMAF,           STAT3&TGFBR
DLL1,           DLL1
EOMES,          IL27R&RUNX3 | RUNX3&TBET
FOXP3,          FOXP3&!GATA3&!RORGT&!STAT3&!TBET | FOXP3&IL2&!IL21&!RORGT | FOXP3&NFAT&STAT5 | !GATA3&!IL21R&!IL6R&STAT5 | !GATA3&!IL21R&!IL6R&TGFBR | !GATA3&!IL21R&!STAT3&STAT5 | !GATA3&!IL21R&!STAT3&TGFBR | !GATA3&!RORGT&SMAD2&!STAT3&!TBET | IL2&!IL21&!RORGT&TGFB | NFAT&!RORGT&SMAD3&!STAT1&STAT5 | NFAT&SMAD3&!STAT1&!STAT3&STAT5
GATA3,          !BCL6&GATA3&!IL29R&!PU1 | !BCL6&!IFNG&IL2&!IL21&IL4&!TBET&!TGFB | !BCL6&IL25R&!IL29R&!PU1&!TBET | !FOXP3&!RORGT&STAT5&!TBET&!TGFB | GATA3&!TBET | STAT6&!TBET
GZMB,           EOMES
IFNAR,          IFNA_e | IFNB_e
IFNA_e,         IFNA_e
IFNBR,          IFNB_e
IFNB_e,         IFNB_e
IFN

In [11]:
sources = []
for node in merged_regulators_dict:
    print(f"{node}: {merged_regulators_dict[node]}")
    if len(merged_regulators_dict[node]) == 1 and merged_regulators_dict[node][0] == node:
        sources.append(node)


APC: ('APC',)
BCL6: ('IL2', 'IL21', 'STAT1', 'STAT3', 'STAT4', 'STAT5', 'TBET', 'TGFB')
CD28: ('APC',)
CD4: ('CD4', 'NOTCH1', 'RUNX3', 'THPOK')
CD8: ('CD8', 'NOTCH1', 'RUNX3', 'TCR', 'THPOK')
CMAF: ('STAT3', 'TGFBR')
DLL1: ('DLL1',)
EOMES: ('IL27R', 'RUNX3', 'TBET')
FOXP3: ('FOXP3', 'GATA3', 'IL2', 'IL21', 'IL21R', 'IL6R', 'NFAT', 'RORGT', 'SMAD2', 'SMAD3', 'STAT1', 'STAT3', 'STAT5', 'STAT6', 'TBET', 'TGFB', 'TGFBR')
GATA3: ('BCL6', 'FOXP3', 'GATA3', 'IFNG', 'IL2', 'IL21', 'IL25R', 'IL29R', 'IL4', 'NFAT', 'PU1', 'RORGT', 'STAT5', 'STAT6', 'TBET', 'TGFB')
GZMB: ('EOMES',)
IFNAR: ('IFNA_e', 'IFNB_e')
IFNA_e: ('IFNA_e',)
IFNBR: ('IFNB_e',)
IFNB_e: ('IFNB_e',)
IFNG: ('BCL6', 'EOMES', 'FOXP3', 'GATA3', 'IFNG', 'IFNG_2', 'IFNG_e', 'IL10', 'IL18R', 'IL21', 'IL4', 'IL9', 'IRAK', 'NFAT', 'NFKB', 'RUNX3', 'STAT3', 'STAT4', 'TBET', 'TGFB', 'proliferation')
IFNGR: ('IFNG', 'IFNGR', 'IFNGR_2', 'IFNG_e', 'NFAT')
IFNGR_2: ('IFNG', 'IFNGR', 'IFNG_2', 'SOCS1')
IFNG_2: ('IFNG', 'IRAK', 'STAT4', 'TBET', 

In [12]:
print(f"{sources=}")
print(len(sources))

sources=['APC', 'DLL1', 'IFNA_e', 'IFNB_e', 'IFNG_e', 'IL10_e', 'IL12_e', 'IL15_e', 'IL18_e', 'IL1_e', 'IL21_e', 'IL23_e', 'IL25_e', 'IL27_e', 'IL29_e', 'IL2_e', 'IL33_e', 'IL36_e', 'IL4_e', 'IL6_e', 'IL7_e', 'TGFB_e']
22
